In [1]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import multilabel_confusion_matrix
import numpy as np
import pandas as pd
import os
import json
import random
import argparse

# General Functions

In [3]:
import os
import json
import random 
import argparse


def read_json(path):
    with open(path, 'r') as file:
        data = json.load(file)
    return data

def write_json(data, path):
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(path, 'w') as file:
        json.dump(data, file, indent=4)

In [4]:
def get_seen_gt(run_id, task_id, folder_path):
    relation_list = []
    for id in range(1, task_id+1):
        # print(id)
        file = folder_path+"/run{0}".format(run_id)+"/task{0}".format(id)+"/test.json"
        relations = [item['relation'] for item in read_json(file)]
        # print(len(relations))
        relation_list.extend(relations)
    return relation_list

In [5]:
def get_false_prediction(preds, gt):
    false_preds = []
    # print(f"pred len:{len(preds)}--- gt len:{len(gt)}")
    for id, pred in enumerate(preds):
        if pred != gt[id]:
            false = {'truth':gt[id], 'pred':pred}
            false_preds.append(false)
    
    return false_preds
        

In [6]:
def return_hallucinated_prediction(preds, predefined_relations):
    hallucinated = []
    predefined_relations = list(set(predefined_relations))
    predefined_relations = [ item.split(':')[-1] for item in predefined_relations]
    # print(len(predefined_relations))
    preds = [pred.split(':')[-1] for pred in preds]
    
    for id, pred in enumerate(preds):
        if pred in predefined_relations:
            continue
        hallucinated.append(pred)

    return hallucinated
            

In [21]:
def seen_predicts(result_folder, gt_folder, relation_folder):
    false_predictions = []
    hallucination = []
    for run_id in range(1,6):
        unique_hallucination = []
        for task_id in range(1,9):
            # print(task_id)
            file_path =  result_folder + "/model{0}/".format(run_id) + "task_{0}_seen_task.json".format(task_id)
            preds = [pred['predict'] for pred in read_json(file_path)]
            gt_relations = get_seen_gt(run_id, task_id, gt_folder)
       
            false_preds = get_false_prediction(preds, gt_relations)
            
            hallucinated = return_hallucinated_prediction(preds, gt_relations)
            falses = {'run_id':run_id, "task_id":task_id, "false":false_preds, 'hallucination':hallucinated, 
                      'count_unique_hallucination': len(list(set(hallucinated))), 'unique_hallucinated_relations':list(set(hallucinated))}
            false_predictions.append(falses)
            unique_hallucination.extend(list(set(hallucinated)))
        hallucination.append({'run_id':run_id, 'unique_hallucination':list(set(unique_hallucination))})
    return false_predictions, hallucination
        

# TACRED

In [24]:
experiments= {
        # 'mas_discovery':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/mas_flan',
#              'mas':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/mas_correct_tacred',
#              'synaptic_intelligenece':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/si_tacred_flan_random',
#              'elastic_weight_consolidation':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/ewc_corrected_tacred_100',
              'baseline':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/baseline'
             }

In [26]:
ground_truth = {'test':'/Users/sefika/phd_projects/llm-catastrophic-re/FSCRE/tacred/5way5shot-test'}
relations = {'relation': '/Users/sefika/phd_projects/llm-catastrophic-re/FSCRE/tacred/relations/'}

In [28]:
for key, value in experiments.items():
    false_predictions, hallucination = seen_predicts(experiments[key], ground_truth['test'], relations['relation'])
    false_pred_path = value+"/"+key+'_false_pred.json'
    hallucination_path = value+"/"+key+'_hallucination_pred.json'
    write_json(false_predictions, false_pred_path)
    write_json(hallucination, hallucination_path)

In [171]:
## Hallucinated predictions

## FewRel

In [175]:
experiments= {'mas_discovery':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/mas_fewrel',
             'mas':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/mas_correct_fewrel',
             'synaptic_intelligenece':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/si_fewrel',
             'elastic_weight_consolidation':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/ewc_corrected_fewrel_100'
             }

In [177]:
ground_truth = {'test':'/Users/sefika/phd_projects/llm-catastrophic-re/FSCRE/fewrel/10way5shot-test'}
relations = {'relation': '/Users/sefika/phd_projects/llm-catastrophic-re/FSCRE/tacred/relations/'}

In [179]:
for key, value in experiments.items():
    false_predictions = seen_predicts(experiments[key], ground_truth['test'], relations['relation'])
    false_pred_path = value+"/"+key+'_false_pred.json'
    write_json(false_predictions, false_pred_path)